# Molecule CV — Model Comparison & DeLong Test

Runs existing architectures, saves predictions to CSV, compares ROC/AUC via DeLong test.

**Runtime:** T4 GPU recommended. ~10 min total.

In [ ]:
# Mount Drive if using Colab
from google.colab import drive
drive.mount('/content/drive')

# Set this to wherever the repo/data lives
DATA_DIR = '/content/drive/MyDrive/Molecule_CV'  # adjust as needed

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, mean_absolute_error
import matplotlib.pyplot as plt

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## Load Data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'bace.csv'))
print(f"Samples: {len(df)}, Columns: {len(df.columns)}")
print(f"Class distribution: {df['Class'].value_counts().to_dict()}")
print(f"pIC50 range: {df['pIC50'].min():.2f} — {df['pIC50'].max():.2f}")

In [ ]:
IMG_DIR = os.path.join(DATA_DIR, 'molecule_images')

def load_images_and_targets(df, img_dir, img_size=(180, 180)):
    """Load molecule images and extract targets."""
    images, pic50s, classes, cids = [], [], [], []
    missing = 0
    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, f"{row['CID']}.png")
        if not os.path.exists(img_path):
            missing += 1
            continue
        img = tf.io.read_file(img_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        images.append(img)
        pic50s.append(row['pIC50'])
        classes.append(row['Class'])
        cids.append(row['CID'])
    if missing:
        print(f"Warning: {missing} images not found")
    return np.array(images), np.array(pic50s), np.array(classes), cids

images, pic50, labels, cids = load_images_and_targets(df, IMG_DIR)
print(f"Loaded {len(images)} images, shape: {images[0].shape}")

In [ ]:
# Same split strategy as existing notebooks: 80/10/10
X_temp, X_test, y_temp, y_test, c_temp, c_test, cid_temp, cid_test = train_test_split(
    images, pic50, labels, cids, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val, c_train, c_val, cid_train, cid_val = train_test_split(
    X_temp, y_temp, c_temp, cid_temp, test_size=0.5, random_state=SEED)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## Define Architectures

All three from the existing notebooks — non-pretrained.

In [ ]:
def build_admet_model(input_shape=(180, 180, 3)):
    """Shi et al. 2019"""
    return keras.Sequential([
        keras.layers.Conv2D(16, (21, 21), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((14, 14), padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

def build_custom_cnn(input_shape=(180, 180, 3)):
    """Multi-layer CNN (test_model_PIC)"""
    return keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(128, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(256, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1)
    ])

def build_toxic_colors(input_shape=(180, 180, 3)):
    """Fernandez et al. 2018"""
    return keras.Sequential([
        keras.layers.Conv2D(12, (16, 16), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.4),
        keras.layers.Flatten(),
        keras.layers.Dense(40, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

MODELS = {
    'ADMET (Shi 2019)': build_admet_model,
    'Custom CNN': build_custom_cnn,
    'Toxic Colors (Fernandez 2018)': build_toxic_colors,
}

## Train & Collect Predictions

In [ ]:
results = {}  # model_name -> {history, predictions, metrics}

for name, builder in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")

    model = builder()
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    epochs = 100 if name != 'Custom CNN' else 50
    batch_size = 128 if name != 'Custom CNN' else 32

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
    preds = model.predict(X_test, verbose=0).flatten()

    results[name] = {
        'history': history.history,
        'predictions': preds,
        'test_loss': test_loss,
        'test_mae': test_mae,
    }
    print(f"\n{name} — Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

## Save Results to CSV

In [ ]:
OUTPUT_DIR = os.path.join(DATA_DIR, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. Predictions per molecule ---
pred_df = pd.DataFrame({'CID': cid_test, 'pIC50_actual': y_test, 'Class': c_test})
for name, res in results.items():
    col = name.split('(')[0].strip().replace(' ', '_').lower()
    pred_df[f'pred_{col}'] = res['predictions']
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions.csv'), index=False)
print(f"Saved predictions.csv ({len(pred_df)} rows)")

# --- 2. Model summary metrics ---
metrics_rows = []
for name, res in results.items():
    auc = roc_auc_score(c_test, res['predictions'])
    metrics_rows.append({
        'model': name,
        'test_mse': res['test_loss'],
        'test_mae': res['test_mae'],
        'auc_roc': auc,
    })
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'model_metrics.csv'), index=False)
print("Saved model_metrics.csv")
print(metrics_df.to_string(index=False))

# --- 3. Training history ---
for name, res in results.items():
    col = name.split('(')[0].strip().replace(' ', '_').lower()
    hist_df = pd.DataFrame(res['history'])
    hist_df.index.name = 'epoch'
    hist_df.to_csv(os.path.join(OUTPUT_DIR, f'history_{col}.csv'))
print("Saved training histories")

## DeLong Test — AUC Comparison

Uses pIC50 predictions as scores for binary classification (Class = active/inactive).  
DeLong test asks: is the AUC difference between two models statistically significant?

In [ ]:
# DeLong test implementation (Sun & Xu, 2014)
# No extra dependencies needed — just numpy + scipy
from scipy.stats import norm

def delong_test(y_true, pred_a, pred_b):
    """Two-sided DeLong test comparing AUCs of two models.

    Returns: auc_a, auc_b, z_stat, p_value
    """
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    pos = y_true == 1
    neg = y_true == 0
    x_p = np.vstack([pred_a[pos], pred_b[pos]])  # (2, m)
    x_n = np.vstack([pred_a[neg], pred_b[neg]])  # (2, n)
    m = x_p.shape[1]
    n = x_n.shape[1]

    # Structural components (placements) via Mann-Whitney U
    # V10[i] = (1/n) * sum(I(x_n < x_p[i]) + 0.5*I(x_n == x_p[i]))
    # V01[j] = (1/m) * sum(I(x_p > x_n[j]) + 0.5*I(x_p == x_n[j]))
    v_10 = np.zeros((2, m))
    v_01 = np.zeros((2, n))
    aucs = np.zeros(2)

    for k in range(2):
        v_10[k] = np.array([
            np.mean((x_n[k] < xi) + 0.5 * (x_n[k] == xi)) for xi in x_p[k]
        ])
        v_01[k] = np.array([
            np.mean((x_p[k] > xj) + 0.5 * (x_p[k] == xj)) for xj in x_n[k]
        ])
        aucs[k] = np.mean(v_10[k])

    # Covariance matrix of AUCs
    s10 = np.cov(v_10)  # (2,2)
    s01 = np.cov(v_01)  # (2,2)
    s = s10 / m + s01 / n

    # Z statistic
    diff = aucs[0] - aucs[1]
    var_diff = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    if var_diff <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = diff / np.sqrt(var_diff)
    p = 2 * norm.sf(abs(z))

    return aucs[0], aucs[1], z, p

In [ ]:
# Run DeLong test for all model pairs
model_names = list(results.keys())
delong_rows = []

for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a, b = model_names[i], model_names[j]
        auc_a, auc_b, z, p = delong_test(
            c_test, results[a]['predictions'], results[b]['predictions']
        )
        delong_rows.append({
            'model_a': a, 'model_b': b,
            'auc_a': round(auc_a, 4), 'auc_b': round(auc_b, 4),
            'z_stat': round(z, 4), 'p_value': p,
        })

# Holm-Bonferroni correction across the k pairwise tests
k = len(delong_rows)
order = sorted(range(k), key=lambda i: delong_rows[i]['p_value'])
running_max = 0.0
p_holm = [None] * k
for rank, idx in enumerate(order):
    adj = (k - rank) * delong_rows[idx]['p_value']
    running_max = max(running_max, adj)
    p_holm[idx] = min(running_max, 1.0)

for row, p_adj in zip(delong_rows, p_holm):
    row['p_holm'] = round(p_adj, 6)
    row['p_value'] = round(row['p_value'], 6)
    row['sig'] = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'ns'
    print(f"{row['model_a']} vs {row['model_b']}: "
          f"AUC {row['auc_a']:.4f} vs {row['auc_b']:.4f}, "
          f"z={row['z_stat']:.3f}, p={row['p_value']:.4f}, "
          f"p_holm={row['p_holm']:.4f} {row['sig']}")

delong_df = pd.DataFrame(delong_rows)
delong_df.to_csv(os.path.join(OUTPUT_DIR, 'delong_results.csv'), index=False)
print(f"\nSaved delong_results.csv")

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(c_test, res['predictions'])
    auc = roc_auc_score(c_test, res['predictions'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Architecture Comparison (Non-Pretrained)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'roc_comparison.png'), dpi=150)
plt.show()
print("Saved roc_comparison.png")

## Summary

In [ ]:
print("\nFiles saved to", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size:,} bytes)")

print("\n--- Model Metrics ---")
print(metrics_df.to_string(index=False))
print("\n--- DeLong Test ---")
print(delong_df.to_string(index=False))